# 04 — Train Prophet Temperature Forecaster

Facebook Prophet for 60-minute ahead temperature forecasting.
Feeds the What-If Simulator for counterfactual cooling scenarios.


In [ ]:
import pandas as pd
from prophet import Prophet
import joblib
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/cooling_control.csv')
df['ds'] = pd.to_datetime(df['timestamp'])
ycol = 'inlet_temp_c' if 'inlet_temp_c' in df.columns else df.columns[1]
df['y'] = df[ycol]
df['cooling_setpoint_c'] = df.get('cooling_setpoint_c', 22.0)
df['hour_of_day'] = df['ds'].dt.hour
df['day_of_week'] = df['ds'].dt.dayofweek
pdf = df[['ds','y','cooling_setpoint_c','hour_of_day','day_of_week']].dropna()
print(f'Training on {len(pdf)} rows, target={ycol}')


In [ ]:
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True)
model.add_regressor('cooling_setpoint_c')
model.add_regressor('hour_of_day')
model.add_regressor('day_of_week')
model.fit(pdf)
joblib.dump(model, '../artifacts/prophet_temp_v1.joblib')
print('Prophet trained and saved.')


## Next Steps
- What-If: increase cooling setpoint → Prophet predicts temp rise → energy saving calculated
- RiskScorer uses forecast deviation as 35% weight in ensemble
